In [1]:
!pip uninstall -y onnxruntime onnxruntime-gpu

!pip install -q onnxruntime-gpu==1.22.0 insightface==0.7.3

import onnxruntime as ort
print(ort.get_available_providers())

Found existing installation: onnxruntime-gpu 1.22.0
Uninstalling onnxruntime-gpu-1.22.0:
  Successfully uninstalled onnxruntime-gpu-1.22.0
['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [4]:
# =========================================================
# ARCFACE EMBEDDING EXTRACTION (GPU ALREADY RUNNING)
# =========================================================

import os
import cv2
import numpy as np
from tqdm import tqdm

# =========================================================
# DATASET PATH
# =========================================================
DATASET_DIR = '/kaggle/input/datasets/jangedoo/utkface-new/UTKFace'

# =========================================================
# GET RECOGNITION MODEL
# =========================================================
rec_model = None

for name, model in app.models.items():
    if 'recognition' in name.lower() or 'w600k' in name.lower():
        rec_model = model
        break

if rec_model is None:
    rec_model = app.models.get('recognition')

print("Recognition backbone ready.")

# =========================================================
# LOAD IMAGES
# =========================================================
image_names = [
    f for f in os.listdir(DATASET_DIR)
    if f.endswith('.jpg')
]

print(f"Total images found: {len(image_names)}")

# =========================================================
# EXTRACT EMBEDDINGS
# =========================================================
embeddings_list = []
labels_list = []

skipped = 0

for img_name in tqdm(image_names, desc="Extracting ArcFace Embeddings"):

    try:
        # UTKFace:
        # [age]_[gender]_[race]_[timestamp].jpg

        parts = img_name.split('_')

        if len(parts) < 3:
            skipped += 1
            continue

        gender = int(parts[1])

        img_path = os.path.join(DATASET_DIR, img_name)

        img = cv2.imread(img_path)

        if img is None:
            skipped += 1
            continue

        # Resize directly to ArcFace input size
        aligned = cv2.resize(img, (112, 112))

        # =================================================
        # TEST TIME AUGMENTATION
        # Original + Flipped
        # =================================================
        emb_views = []

        for view in [aligned, cv2.flip(aligned, 1)]:

            feat = rec_model.get_feat(view).flatten()

            # L2 Normalize
            feat = feat / (np.linalg.norm(feat) + 1e-8)

            emb_views.append(feat)

        # Average embeddings
        final_emb = np.mean(emb_views, axis=0)

        # Final normalize
        final_emb /= (np.linalg.norm(final_emb) + 1e-8)

        embeddings_list.append(final_emb)
        labels_list.append(gender)

    except Exception:
        skipped += 1
        continue

# =========================================================
# CONVERT TO NUMPY
# =========================================================
X_arr = np.array(embeddings_list, dtype=np.float32)
y_arr = np.array(labels_list, dtype=np.int32)

print("\n==============================")
print("EXTRACTION COMPLETE")
print("==============================")
print(f"Embeddings Shape : {X_arr.shape}")
print(f"Labels Shape     : {y_arr.shape}")
print(f"Skipped Images   : {skipped}")

# =========================================================
# SAVE FILES
# =========================================================
np.save('/kaggle/working/arcface_embeddings_tta.npy', X_arr)
np.save('/kaggle/working/gender_labels_tta.npy', y_arr)

print("\nSaved:")
print("/kaggle/working/arcface_embeddings_tta.npy")
print("/kaggle/working/gender_labels_tta.npy")

Recognition backbone ready.
Total images found: 23708


Extracting ArcFace Embeddings: 100%|██████████| 23708/23708 [05:55<00:00, 66.74it/s]



EXTRACTION COMPLETE
Embeddings Shape : (23708, 512)
Labels Shape     : (23708,)
Skipped Images   : 0

Saved:
/kaggle/working/arcface_embeddings_tta.npy
/kaggle/working/gender_labels_tta.npy


In [5]:
X_verify = np.array(embeddings_list, dtype=np.float32)
y_verify = np.array(labels_list, dtype=np.int32)

print("=== KAGGLE MEMORY VERIFICATION ===")
print(f"Features Array Shape: {X_verify.shape}")
print(f"Labels Array Shape:   {y_verify.shape}")

if X_verify.shape[0] == 0:
    print("CRITICAL: The lists are empty! Check your loop logic or paths.")
else:
    # Save files only if they contain real data
    np.save('arcface_embeddings_tta.npy', X_verify)
    np.save('gender_labels_tta.npy', y_verify)
    print("Status: Success! Files written to disk with real samples.")

=== KAGGLE MEMORY VERIFICATION ===
Features Array Shape: (23708, 512)
Labels Array Shape:   (23708,)
Status: Success! Files written to disk with real samples.
